# Workflow Completo: ERA5 Download, Preprocess, and WAM2Layers Execution

Este notebook automatiza el flujo completo de:
1. Descarga de datos ERA5 desde CDS API
2. Preprocesamiento y descomposición de datos
3. Generación del archivo de configuración YAML
4. Ejecución de wam2layers
5. Limpieza de datos temporales


## 1. Importar librerías necesarias

In [7]:
import os
import shutil
import subprocess
from pathlib import Path
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import xarray as xr
import yaml

import cdsapi

## 2. Definir parámetros y rutas de trabajo

In [8]:
# Configuración de fechas y rutas
START_DATE = "2023-01-01"  # Fecha de inicio en formato YYYY-MM-DD
END_DATE = "2023-01-02"    # Fecha de fin en formato YYYY-MM-DD

# Parsear fechas
start = pd.to_datetime(START_DATE)
end = pd.to_datetime(END_DATE)

# Extraer año
year = start.year

# Generar rango de fechas
datelist = pd.date_range(START_DATE, END_DATE)
months = list(set([str(d.month).zfill(2) for d in datelist]))
months.sort()

# Diccionarios de días y fechas por mes
days_all = {}
dates = []
for month in months:
    month_days = datelist[datelist.month == int(month)]
    days_all[month] = [str(d.day).zfill(2) for d in month_days]
    month_str = month
    start_day = min(month_days).strftime("%d")
    end_day = max(month_days).strftime("%d")
    dates.append(f"{year}-{month}-{start_day}/to/{year}-{month}-{end_day}")

print(f"Período: {START_DATE} a {END_DATE}")
print(f"Meses a descargar: {months}")
print(f"Días por mes: {days_all}")

# Rutas generales
target_dir = "."
preprocessed_dir = "./preprocessed_data"
output_dir = "./output_data"
yaml_config_name = "workflow_config.yaml"

# Configuración de descarga
skip_exist = True
area = None  # None para global, o [N, W, S, E]
grid = [0.25, 0.25]

# Variables a descargar
ml_variables = {
    "u": 131,
    "v": 132,
    "q": 133,
}

surface_variables = {
    "tp": "total_precipitation",
    "e": "evaporation",
    "sp": "surface_pressure",
    "tcw": "total_column_water",
}

# Niveles de modelo
levels = "20/40/60/80/90/95/100/105/110/115/120/123/125/128/130/131/132/133/134/135/136/137"

# Tiempos (24 horas)
times = [f"{h:02d}:00" for h in range(24)]

print(f"\nConfiguraciones:")
print(f"- Directorio de salida preprocessado: {preprocessed_dir}")
print(f"- Directorio de output: {output_dir}")
print(f"- Variables ML: {list(ml_variables.keys())}")
print(f"- Variables superficie: {list(surface_variables.keys())}")

Período: 2023-01-01 a 2023-01-02
Meses a descargar: ['01']
Días por mes: {'01': ['01', '02']}

Configuraciones:
- Directorio de salida preprocessado: ./preprocessed_data
- Directorio de output: ./output_data
- Variables ML: ['u', 'v', 'q']
- Variables superficie: ['tp', 'e', 'sp', 'tcw']


## 3. Descargar datos ERA5

In [9]:
print("="*80)
print("INICIANDO DESCARGA DE DATOS ERA5")
print("="*80)

# Inicializar cliente CDS
c = cdsapi.Client(timeout=600, quiet=False)

ind = 0
for month in months:
    outfolder = Path(target_dir) / f"{year}" / f"{month}"
    outfolder.mkdir(exist_ok=True, parents=True)
    
    this_days = days_all[month]
    
    # Descargar variables de superficie
    print(f"\n--- Descargando variables de superficie para mes {month} ---")
    for variable, long_name in surface_variables.items():
        outfile = f"ERA5_{year}-{month}_{variable}.nc"
        filepath = outfolder / outfile
        
        if filepath.exists() and skip_exist:
            print(f"✓ {filepath} ya existe, saltando.")
        else:
            try:
                print(f"Descargando {variable}...")
                c.retrieve(
                    "reanalysis-era5-single-levels",
                    {
                        "product_type": "reanalysis",
                        "variable": long_name,
                        "year": f"{year}",
                        "month": month,
                        "day": this_days,
                        "time": times,
                        "grid": grid,
                        "format": "netcdf",
                    },
                    str(filepath),
                )
                print(f"✓ Descargado: {filepath}")
            except Exception as e:
                print(f"✗ Error descargando {variable} mes {month}: {e}")
    
    # Descargar variables de modelo (3D)
    print(f"\n--- Descargando variables de modelo para mes {month} ---")
    for variable, param in ml_variables.items():
        outfile = f"ERA5_{year}-{month}_ml_{variable}.nc"
        filepath = outfolder / outfile
        
        if filepath.exists() and skip_exist:
            print(f"✓ {filepath} ya existe, saltando.")
        else:
            try:
                print(f"Descargando ml_{variable}...")
                c.retrieve(
                    "reanalysis-era5-complete",
                    {
                        "time": times,
                        "dates": dates[ind],
                        "levelist": levels,
                        "param": param,
                        "class": "ea",
                        "expver": "1",
                        "levtype": "ml",
                        "stream": "oper",
                        "type": "an",
                        "format": "netcdf",
                        "grid": grid,
                    },
                    str(filepath),
                )
                print(f"✓ Descargado: {filepath}")
            except Exception as e:
                print(f"✗ Error descargando ml_{variable} mes {month}: {e}")
    
    ind += 1

print("\n" + "="*80)
print("✓ DESCARGA COMPLETADA")
print("="*80)

INICIANDO DESCARGA DE DATOS ERA5

--- Descargando variables de superficie para mes 01 ---
✓ 2023/01/ERA5_2023-01_tp.nc ya existe, saltando.
✓ 2023/01/ERA5_2023-01_e.nc ya existe, saltando.
✓ 2023/01/ERA5_2023-01_sp.nc ya existe, saltando.
✓ 2023/01/ERA5_2023-01_tcw.nc ya existe, saltando.

--- Descargando variables de modelo para mes 01 ---
✓ 2023/01/ERA5_2023-01_ml_u.nc ya existe, saltando.
✓ 2023/01/ERA5_2023-01_ml_v.nc ya existe, saltando.
✓ 2023/01/ERA5_2023-01_ml_q.nc ya existe, saltando.

✓ DESCARGA COMPLETADA


## 4. Preprocesar datos descargados

In [ ]:
print("="*80)
print("INICIANDO PREPROCESAMIENTO Y DESCOMPOSICIÓN")
print("="*80)

# Crear directorio de preprocesamiento
output_base_dir = Path(preprocessed_dir) / "descompuestos"
output_base_dir.mkdir(exist_ok=True, parents=True)

# Variables a procesar (combina ML y superficie)
surface_vars = list(surface_variables.keys())
ml_vars = list(ml_variables.keys())


def descomponer_archivos():
    """Descompone archivos mensuales en archivos diarios usando procesamiento por chunks

    Estrategia:
    - Procesar primero variables de superficie (ligeras) y eliminar sus archivos mensuales para liberar disco y memoria
    - Luego procesar variables de modelo (ML) por día, y dentro del día en pequeños batchs horarios
    - Usar xarray+dask (chunks={'time':1}) para lazy loading y llamar a .compute() solo en pequeños subconjuntos
    """
    for month in months:
        mes_str = f"{month:0>2}"
        mes_path = Path(target_dir) / f"{year}" / mes_str

        if not mes_path.exists():
            print(f"✗ Carpeta {mes_path} no encontrada.")
            continue

        # Crear carpeta de salida para el mes
        output_dir = output_base_dir / mes_str
        output_dir.mkdir(exist_ok=True, parents=True)

        print(f"\n--- Procesando mes {mes_str} ---")

        # 1) Procesar variables de superficie y eliminar el archivo mensual para ahorrar memoria
        for var in surface_vars:
            archivo_mensual = mes_path / f"ERA5_{year}-{mes_str}_{var}.nc"
            if not archivo_mensual.exists():
                print(f"  ℹ Archivo {archivo_mensual.name} no encontrado, saltando.")
                continue

            try:
                print(f"  Procesando superficie: {var} (día a día)...")
                with xr.open_dataset(archivo_mensual, chunks={'time': 1}, decode_cf=True) as ds:
                    ds = ds.drop_vars(['expver', 'number'], errors='ignore')
                    if 'ps' in ds.data_vars:
                        ds = ds.rename({'ps': 'sp'})
                    if 'valid_time' in ds.coords:
                        ds = ds.rename({'valid_time': 'time'})

                    time_coords = ds['time'].values
                    unique_dates = np.unique(np.array([str(pd.Timestamp(t).date()) for t in time_coords]))

                    for dia_idx, dia_str in enumerate(unique_dates, 1):
                        try:
                            ds_dia = ds.sel(time=ds['time'].dt.date == np.datetime64(dia_str))
                            ds_dia = ds_dia.compute()

                            for var_name in ds_dia.data_vars:
                                ds_dia[var_name] = ds_dia[var_name].astype(np.float64)

                            if 'model_level' in ds_dia.dims:
                                ds_dia = ds_dia.transpose('time', 'level', 'latitude', 'longitude')
                                ds_dia = ds_dia.assign_coords(longitude=(((ds_dia.longitude + 180) % 360) - 180))
                            else:
                                ds_dia = ds_dia.transpose('time', 'latitude', 'longitude')

                            output_filename = f"ERA5_{dia_str}_{var}.nc"
                            output_path = output_dir / output_filename
                            ds_dia.to_netcdf(output_path)

                            del ds_dia
                            gc.collect()

                            if dia_idx % max(1, len(unique_dates)//3) == 0:
                                print(f"    ✓ {dia_idx}/{len(unique_dates)} días procesados para {var}")

                        except Exception as e:
                            print(f"    ✗ Error procesando día {dia_str} de {var}: {e}")

                del ds
                gc.collect()
                print(f"  ✓ {var} procesado y liberado")

            except Exception as e:
                print(f"  ✗ Error procesando {var}: {type(e).__name__}: {str(e)[:120]}")

        # 2) Procesar variables de modelo (ML) por día, en pequeños batchs horarios
        for ml in ml_vars:
            archivo_mensual = mes_path / f"ERA5_{year}-{mes_str}_ml_{ml}.nc"
            if not archivo_mensual.exists():
                print(f"  ℹ Archivo {archivo_mensual.name} no encontrado, saltando.")
                continue

            try:
                print(f"  Procesando ML: ml_{ml} (por día completo)...")
                with xr.open_dataset(archivo_mensual, chunks={'time': 1}, decode_cf=True) as ds:
                    ds = ds.drop_vars(['expver', 'number'], errors='ignore')
                    if 'valid_time' in ds.coords:
                        ds = ds.rename({'valid_time': 'time'})
                    if 'ps' in ds.data_vars:
                        ds = ds.rename({'ps': 'sp'})
                    if 'model_level' in ds.dims:
                        ds = ds.rename({'model_level': 'level'})

                    # Convertir tiempo a datetime si es cftime
                    if hasattr(ds['time'].values[0], 'calendar'):
                        ds['time'] = pd.to_datetime(ds['time'].values)

                    time_coords = ds['time'].values
                    unique_dates = np.unique(np.array([str(pd.Timestamp(t).date()) for t in time_coords]))

                    for dia_idx, dia_str in enumerate(unique_dates, 1):
                        try:
                            # índices de tiempo para el día
                            indices = np.where(np.array([str(pd.Timestamp(t).date()) for t in time_coords]) == dia_str)[0]
                            if len(indices) == 0:
                                continue

                            # Procesar el día completo sin batching para evitar pérdida de horas
                            ds_dia = ds.isel(time=indices).compute()

                            for var_name in ds_dia.data_vars:
                                ds_dia[var_name] = ds_dia[var_name].astype(np.float64)

                            # Transponer para consistencia
                            if 'level' in ds_dia.dims:
                                ds_dia = ds_dia.transpose('time', 'level', 'latitude', 'longitude')
                                ds_dia = ds_dia.assign_coords(longitude=(((ds_dia.longitude + 180) % 360) - 180))
                            else:
                                ds_dia = ds_dia.transpose('time', 'latitude', 'longitude')

                            output_filename = f"ERA5_{dia_str}_ml_{ml}.nc"
                            output_path = output_dir / output_filename
                            ds_dia.to_netcdf(output_path)

                            if dia_idx % max(1, len(unique_dates)//3) == 0:
                                print(f"    ✓ {dia_idx}/{len(unique_dates)} días procesados para ml_{ml} ({len(indices)} horas)")

                        except Exception as e:
                            print(f"    ✗ Error procesando día {dia_str} de ml_{ml}: {e}")

                    del ds
                    gc.collect()

                print(f"  ✓ ml_{ml} procesado y liberado")

            except Exception as e:
                print(f"  ✗ Error con ml_{ml}: {type(e).__name__}: {str(e)[:120]}")

    print(f"\n--- Mes {mes_str} completo ---")


# Ejecutar
descomponer_archivos()

print("\n" + "="*80)
print("✓ PREPROCESAMIENTO COMPLETADO")
print("="*80)
print(f"Datos preprocesados guardados en: {output_base_dir}")

INICIANDO PREPROCESAMIENTO Y DESCOMPOSICIÓN

--- Procesando mes 01 ---
  Procesando superficie: tp (día a día)...
  ✗ Error procesando tp: NameError: name 'time_coder' is not defined
  Procesando superficie: e (día a día)...
  ✗ Error procesando e: NameError: name 'time_coder' is not defined
  Procesando superficie: sp (día a día)...
  ✗ Error procesando sp: NameError: name 'time_coder' is not defined
  Procesando superficie: tcw (día a día)...
  ✗ Error procesando tcw: NameError: name 'time_coder' is not defined
  Procesando ML: ml_u (por día completo)...
    ✓ 10/31 días procesados para ml_u (24 horas)


## 5. Crear archivo de configuración YAML

In [4]:
print("="*80)
print("CREANDO ARCHIVO DE CONFIGURACIÓN YAML")
print("="*80)

# Crear estructura YAML manualmente para asegurar que las fechas estén entre comillas
yaml_path = Path(yaml_config_name)

yaml_text = f"""# Output
preprocessed_data_folder: {preprocessed_dir}
output_folder: {output_dir}
output_frequency: "1d"

# Preprocessing
filename_template: "{'preprocessed_data/descompuestos/{month:02}/ERA5_{year}-{month:02d}-{day:02d}{levtype}_{variable}.nc' }"
preprocess_start_date: "{start.strftime('%Y-%m-%dT00:00')}"
preprocess_end_date: "{end.strftime('%Y-%m-%dT00:00')}"
level_type: model_levels
levels: [20,40,60,80,90,95,100,105,110,115,120,123,125,128,130,131,132,133,134,135,136,137]
input_frequency: "1h"

# Tracking
tracking_direction: backward
tracking_domain: null
periodic_boundary: False
tracking_start_date: "{start.strftime('%Y-%m-%dT00:00')}"
tracking_end_date: "{end.strftime('%Y-%m-%dT00:00')}"
timestep: 600
kvf: 3

# Tagging
tagging_start_date: "{start.strftime('%Y-%m-%dT00:00')}"
tagging_end_date: "{end.strftime('%Y-%m-%dT00:00')}"
restart: False
tagging_region: shapes/sinu.nc
target_frequency: "10min"
"""

with open(yaml_path, 'w') as f:
    f.write(yaml_text)

print(f"✓ Archivo YAML creado: {yaml_path}")
print(f"\nContenido del archivo YAML:")
print("-" * 80)
with open(yaml_path, 'r') as f:
    print(f.read())
print("-" * 80)

CREANDO ARCHIVO DE CONFIGURACIÓN YAML
✓ Archivo YAML creado: workflow_config.yaml

Contenido del archivo YAML:
--------------------------------------------------------------------------------
# Output
preprocessed_data_folder: ./preprocessed_data
output_folder: ./output_data
output_frequency: "1d"

# Preprocessing
filename_template: "preprocessed_data/descompuestos/{month:02}/ERA5_{year}-{month:02d}-{day:02d}{levtype}_{variable}.nc"
preprocess_start_date: "2023-01-01T00:00"
preprocess_end_date: "2023-01-02T00:00"
level_type: model_levels
levels: [20,40,60,80,90,95,100,105,110,115,120,123,125,128,130,131,132,133,134,135,136,137]
input_frequency: "1h"

# Tracking
tracking_direction: backward
tracking_domain: null
periodic_boundary: False
tracking_start_date: "2023-01-01T00:00"
tracking_end_date: "2023-01-02T00:00"
timestep: 600
kvf: 3

# Tagging
tagging_start_date: "2023-01-01T00:00"
tagging_end_date: "2023-01-02T00:00"
restart: False
tagging_region: shapes/sinu.nc
target_frequency: "10m

## 6. Ejecutar wam2layers con el archivo YAML

In [5]:
print("="*80)
print("EJECUTANDO WAM2LAYERS")
print("="*80)

# Comando a ejecutar
wam2layers_cmd = ["wam2layers", "preprocess", "era5", str(yaml_path)]

try:
    print(f"Ejecutando: {' '.join(wam2layers_cmd)}")
    print("-" * 80)
    
    # Ejecutar el comando
    result = subprocess.run(
        wam2layers_cmd,
        capture_output=True,
        text=True,
        check=False
    )
    
    # Mostrar salida
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
    
    if result.returncode != 0:
        print(f"\n✗ wam2layers finalizó con código de error: {result.returncode}")
    else:
        print("\n✓ wam2layers ejecutado exitosamente")
        
except Exception as e:
    print(f"✗ Error ejecutando wam2layers: {e}")

print("="*80)
print("✓ EJECUCIÓN DE WAM2LAYERS COMPLETADA")
print("="*80)

EJECUTANDO WAM2LAYERS
Ejecutando: wam2layers preprocess era5 workflow_config.yaml
--------------------------------------------------------------------------------
STDERR: Welcome to WAM2layers.
Starting preprocessing ERA5 data.
/home/unal/anaconda3/envs/wamenv/lib/python3.13/site-packages/wam2layers/preprocessing/shared.py:27: FutureWarning: cftime_range() is deprecated, please use xarray.date_range(..., use_cftime=True) instead.
  return xr.cftime_range(
2023-01-01 00:00:00
/home/unal/anaconda3/envs/wamenv/lib/python3.13/site-packages/wam2layers/preprocessing/era5.py:76: FutureWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  da = xr.open_dataarray(filepath, use_cftime=True)
Traceback (most recent call last):
  File "/home/unal/anaconda3/envs/wam

## 7. Eliminar datos temporales y conservar solo los outputs

In [ ]:
print("="*80)
print("LIMPIANDO DATOS TEMPORALES")
print("="*80)

# Directorios y archivos a eliminar
paths_to_delete = [
    Path(target_dir) / f"{year}",  # Datos descargados originales
    output_base_dir,  # Directorio de preprocesamiento
]

for path in paths_to_delete:
    if path.exists():
        try:
            if path.is_dir():
                shutil.rmtree(path)
                print(f"✓ Directorio eliminado: {path}")
            else:
                path.unlink()
                print(f"✓ Archivo eliminado: {path}")
        except Exception as e:
            print(f"✗ Error eliminando {path}: {e}")
    else:
        print(f"ℹ {path} no existe")

print("\n" + "="*80)
print("RESUMEN FINAL")
print("="*80)
print(f"✓ Datos descargados eliminados")
print(f"✓ Datos preprocesados eliminados")
print(f"✓ Outputs de wam2layers conservados en: {output_dir}")
print(f"✓ Archivo de configuración conservado: {yaml_path}")

# Mostrar contenido del directorio de output
if Path(output_dir).exists():
    output_files = list(Path(output_dir).rglob("*"))
    print(f"\nArchivos generados en {output_dir}:")
    for f in output_files[:20]:  # Mostrar primeros 20 archivos
        if f.is_file():
            print(f"  - {f.relative_to(output_dir)}")
    if len(output_files) > 20:
        print(f"  ... y {len(output_files) - 20} archivos más")

print("="*80)
print("✓✓✓ WORKFLOW COMPLETADO EXITOSAMENTE ✓✓✓")
print("="*80)